# Prefix bridge, two stage-2 paths: carry vs reset, from one stage 1

```
"Thanos: <English sentence>"  →  ModernBERT-base + LoRA  →  bridge (MLP 768 → 2560)  →  one vector per token   [GPU 0]
                                                                                              │
[<bos>] + [prefix vectors] + target  →  Gemma 3 4B IT (4-bit) + LoRA  →  the same meaning in Thanos's voice  <end_of_turn>   [GPU 1]
```

The encoder reads the English sentence; a 2-layer MLP bridge turns each encoder token into a vector in Gemma's own input-embedding space; these vectors are placed after `<bos>` as a **prefix**, and Gemma reads them with its own pretrained attention. There is no text prompt: the sentence reaches Gemma only through the encoder. Trained: encoder LoRA, bridge (≈ 8.5 M parameters), decoder LoRA — **each in its own stage** (this notebook's difference from `9-prefix-bridge/`):

One stage 1 (encoder LoRA + bridge + Gemma LoRA, reconstruction), then **stage 2 twice from the same stage-1 weights**, Gemma's LoRA trained alone (encoder and bridge frozen):

| Path | Gemma's LoRA at the start of stage 2 |
|---|---|
| **carry** | the stage-1 LoRA (it learned to read the bridge and to copy): transfer learning |
| **reset** | zero (Gemma as released; it still reads the bridge, R15): learns the voice without a copy habit |

Each path: early stopping on `validation_daily`, scores on all held-out splits, the eleven chosen sentences, and eight sampled variants of one sentence. A final cell puts both paths side by side. Each path's Gemma LoRA is saved separately; a path that has finished is skipped on a rerun (its results are reloaded), so a kernel death costs at most one path.

| Stage | Data | Encoder input | Pairs |
|---|---|---|---|
| **1. Reconstruction** | English side of Hinglish-TOP (Apache-2.0) and PHINC (CC BY 4.0) | `Plain: <English>` → the same sentence | `CORPUS_TOTAL` in Cell 3 (12,000) |
| **2. Thanos voice, ×2** | `data/thanos_v9/` (the user's style; the user's own lines weighted ×3) | `Thanos: <English>` | up to 6 epochs per path; early stopping on `validation_daily` |

Earlier runs: `LEDGER.md` (R12–R15).


In [ ]:
# Cell 0 — Hugging Face token (Gemma 3 is gated). Clear it before saving or sharing the notebook.

HF_TOKEN = ""

## Cell 1 — Setup, pinned revisions

Two GPUs (Kaggle: *GPU T4 × 2*). Models and both corpora are pinned to the revision found now.

In [ ]:
# Cell 1 — Packages, settings, pinned revisions

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q --force-reinstall --no-deps "transformers==5.17.0" "peft==0.21.2"   # forced: an earlier broken install was left behind
!pip install -q -U bitsandbytes

import gc, json, os, random, re, statistics, time, urllib.error, urllib.request
from collections import Counter

import pandas as pd
import peft
import torch
import torch.nn as nn
import transformers
from huggingface_hub import HfApi, login

ENCODER, DECODER = "answerdotai/ModernBERT-base", "google/gemma-3-4b-it"
TOP_REPO, PHINC_REPO = "rvv-karma/English-Hinglish-TOP", "veezbo/phinc"
BRANCH = "https://raw.githubusercontent.com/skamalj/llms-the-hard-way/hindi-decoder-compare/project4-hindi-decoder/data/"
DATA = BRANCH + "thanos_v9/"
MAX_NEW_TOKENS = 64
DEMO_INPUTS = [                              # translated in the final cell; kept out of every training set
    "I worked all weekend to fix the bug, and nobody noticed.",
    "We can still win this game if we don't give up in the second half.",
    "My brother borrowed my car and returned it with an empty tank.",
    "I'm not angry that you lied; I'm disappointed that you thought you had to.",
    "The project is three weeks late, but the client still wants it by Friday.",
    "I trained for a year, and I missed the medal by one second.",
    # the user's manual test (kept out of all training)
    "I'll be late to the office because of traffic.",
    "I'm running 10 minutes late. Please start without me.",
    "I'm really hungry. Let's get something to eat.",
    "I'm exhausted after today's work. I need some rest.",
    "I'm stuck in a meeting. I'll call you when I'm free.",
]
SEED = 42
DTYPE = torch.float32
WORK = "/kaggle/working"

assert torch.cuda.device_count() >= 2, "this notebook needs two GPUs (Kaggle: GPU T4 x2)"
ENC_DEVICE, DEC_DEVICE = torch.device("cuda:0"), torch.device("cuda:1")
HF_TOKEN = HF_TOKEN.strip() or None
assert HF_TOKEN, "Gemma 3 is gated: paste a Hugging Face token in Cell 0"
login(HF_TOKEN)
api = HfApi()
REV = {r: api.model_info(r, token=HF_TOKEN).sha for r in (ENCODER, DECODER)}
REV.update({r: api.dataset_info(r).sha for r in (TOP_REPO, PHINC_REPO)})
assert (transformers.__version__, peft.__version__) == ("5.17.0", "0.21.2"), (
    f"pinned versions not active (transformers {transformers.__version__}, peft {peft.__version__}): restart the kernel "
    "and run from the top")
print({"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__,
       "gpus": [torch.cuda.get_device_name(i) for i in range(2)], "revisions": REV})

## Cell 2 — Thanos data

`data/thanos_v9/`: English sentence → the same meaning in a Thanos-inspired English voice. One JSON object per line with `id`, `source_id`, `persona_id`, `source`, `target`, optional `references`. Encoder input `"Thanos: <English>"`; target the reference line.

Splits: `train`; **`validation_daily`** (the user's everyday held-out set: early stopping and the main score); `validation` (the earlier curated interpretive set, a reference score only); `test` (in the dramatic voice).

In [ ]:
# Cell 2 — Thanos data: train / validation / test

def fetch_jsonl(name):
    return [json.loads(l) for l in urllib.request.urlopen(DATA + name).read().decode("utf-8").splitlines() if l]


MANIFEST = json.load(urllib.request.urlopen(DATA + "manifest.json"))
NAME = {p: v["name"] for p, v in MANIFEST["personas"].items()}


def persona_record(r):
    pid = r.get("persona_id", "thanos")
    return {**r, "persona": pid, "references": r.get("references") or [r["target"]],
            "enc_input": f"{NAME[pid]}: {r['source']}"}


SPLITS = {s: [persona_record(r) for r in fetch_jsonl(f"{s}.jsonl")]
          for s in ("train", "validation_daily", "validation", "test")}
TRAIN, VAL = SPLITS["train"], SPLITS["validation_daily"]       # early stopping on the user's everyday set
EVAL_SPLITS = ["validation_daily", "validation", "test"]
print({s: len(rs) for s, rs in SPLITS.items()}, "| data:", DATA)
print("first validation pair:", VAL[0]["source"], "->", VAL[0]["target"])

## Cell 3 — Public corpora for stage 1: download, clean, guard against leakage, sample

Stage 1 is **reconstruction**: only the corpora's *English* side is used, each sentence as both input and target ("carry this sentence through the bridge intact").

- **Hinglish-TOP:** keep pairs with 3–30 words on both sides (the pair filter selects clean, ordinary English).
- **PHINC:** remove @handles, links and #hashtags, then the same length rules.
- Drop sentences that resemble any Thanos evaluation sentence (word overlap ≥ 0.6); de-duplicate.
- Hold out 200 sentences from each corpus; the training pool is shuffled and cut to `CORPUS_TOTAL` (default 12,000: a 6,000-sentence stage 1 stopped short of reconstruction, held-out loss 2.99 and chrF 23.1, run R11; reconstruction clicks between steps 500 and 1,000).

In [ ]:
# Cell 3 — Corpora

def download(repo, path, name):
    url = f"https://huggingface.co/datasets/{repo}/resolve/{REV[repo]}/{path}"
    local = f"/tmp/{name}"
    if not os.path.exists(local):
        urllib.request.urlretrieve(url, local)
    return local


top = pd.read_parquet(download(TOP_REPO, "data/train-00000-of-00001.parquet", "top_train.parquet"))
phinc = pd.read_csv(download(PHINC_REPO, "hinglish_english_parallel_corpus.csv", "phinc.csv"))
DEVA = re.compile(r"[ऀ-ॿ]")


def clean_tweet(t):
    t = re.sub(r"https?://\S+|www\.\S+", " ", str(t))
    t = re.sub(r"[@#]\w+", " ", t)
    return re.sub(r"\s+", " ", t).strip(" .…:-")


def usable(en, hi):
    ne, nh = len(en.split()), len(hi.split())
    return 3 <= ne <= 30 and 3 <= nh <= 30 and not DEVA.search(hi) and 0.5 <= nh / ne <= 2.0


pairs = {"top_human": [], "top_synthetic": [], "phinc": []}
for en, hi, gen in zip(top["en"], top["hi_en"], top["generated_by"]):
    en, hi = str(en).strip(), str(hi).strip()
    if usable(en, hi):
        pairs["top_human" if gen == "human" else "top_synthetic"].append((en, hi))
for hi, en in zip(phinc["Sentence"], phinc["English_Translation"]):
    en, hi = clean_tweet(en), clean_tweet(hi)
    if usable(en, hi):
        pairs["phinc"].append((en, hi))

words = lambda t: set(re.findall(r"[a-z0-9']+", t.lower()))
EVAL_WORDS = [words(r["source"]) for s in SPLITS if s != "train" for r in SPLITS[s]] + [words(t) for t in DEMO_INPUTS]
pairs = {k: [(en, en) for en, _ in v] for k, v in pairs.items()}      # reconstruction: English in, same English out


def leaks(en):
    w = words(en)
    return any(len(w & e) / len(w | e) >= 0.6 for e in EVAL_WORDS if w | e)


rng = random.Random(SEED)
for k in pairs:
    rng.shuffle(pairs[k])
seen, dropped = set(), Counter()


def take(items, n):
    """The first n usable pairs of a shuffled list: not seen before, not resembling a Thanos evaluation or demo sentence."""
    out = []
    for en, hi in items:
        if len(out) >= n:
            break
        key = en.lower()
        if key in seen:
            dropped["duplicate"] += 1
        elif leaks(en):
            dropped["resembles an evaluation or demo sentence"] += 1
        else:
            seen.add(key)
            out.append((en, hi))
    return out


CORPUS_TOTAL = 12000                        # stage-1 sentences; 0 skips stage 1
PHINC_MAX = CORPUS_TOTAL // 2              # balance: at most half from PHINC, the rest Hinglish-TOP
HELD = [("top", *p) for p in take(pairs["top_human"], 200)] + [("phinc", *p) for p in take(pairs["phinc"], 200)]
train_pairs = [("top", *p) for p in take(pairs["top_human"][200:], len(pairs["top_human"]))]
train_pairs += [("phinc", *p) for p in take(pairs["phinc"][200:], PHINC_MAX)]
train_pairs += [("top", *p) for p in take(pairs["top_synthetic"], max(0, CORPUS_TOTAL - len(train_pairs)))]
rng.shuffle(train_pairs)
train_pairs = train_pairs[:CORPUS_TOTAL]
print({k: len(v) for k, v in pairs.items()}, "usable before sampling | dropped while sampling", dict(dropped))


def corpus_record(src, en, hi):
    return {"corpus": src, "source": en, "target": hi, "enc_input": f"Plain: {en}", "references": [hi]}


CORPUS_TRAIN = [corpus_record(*p) for p in train_pairs]
del top, phinc, pairs                        # the full corpus tables are no longer needed: free the memory
gc.collect()
CORPUS_HELD = [corpus_record(*p) for p in HELD]
print(f"stage 1 training pairs {len(CORPUS_TRAIN)} {dict(Counter(r['corpus'] for r in CORPUS_TRAIN))} | "
      f"held out {len(CORPUS_HELD)}")
for r in CORPUS_TRAIN[:4]:
    print(f"  {r['enc_input']}  ->  {r['target']}")

## Cell 4 — Models

In [ ]:
# Cell 4 — ModernBERT on GPU 0; Gemma 3 4B IT (4-bit) on GPU 1

from transformers import AutoConfig, AutoModel, AutoTokenizer, BitsAndBytesConfig, Gemma3ForConditionalGeneration

used = {d: torch.cuda.memory_allocated(d) / 2**30 for d in (ENC_DEVICE, DEC_DEVICE)}
if max(used.values()) > 0.5:
    raise RuntimeError("GPU memory already in use: restart the kernel and run all cells once, from the top.")

enc_tok = AutoTokenizer.from_pretrained(ENCODER, revision=REV[ENCODER])
enc_cfg = AutoConfig.from_pretrained(ENCODER, revision=REV[ENCODER])
if hasattr(enc_cfg, "reference_compile"):
    enc_cfg.reference_compile = False
encoder = AutoModel.from_pretrained(ENCODER, revision=REV[ENCODER], config=enc_cfg, dtype=DTYPE,
                                    attn_implementation="sdpa").to(ENC_DEVICE)
dec_tok = AutoTokenizer.from_pretrained(DECODER, revision=REV[DECODER], token=HF_TOKEN)
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=DTYPE)
decoder = Gemma3ForConditionalGeneration.from_pretrained(DECODER, revision=REV[DECODER], dtype=DTYPE, token=HF_TOKEN,
                                                         quantization_config=quant, device_map={"": DEC_DEVICE.index},
                                                         attn_implementation="eager")
START = dec_tok.bos_token_id
EOS = dec_tok.convert_tokens_to_ids("<end_of_turn>")
eos = decoder.generation_config.eos_token_id
STOPS = sorted(set((eos if isinstance(eos, list) else [eos]) + [EOS]))
PAD = dec_tok.pad_token_id
D_ENC, D_DEC = encoder.config.hidden_size, decoder.config.text_config.hidden_size
LAYERS = next(m for n, m in decoder.named_modules() if n.endswith("language_model.layers"))
N_LAYERS = len(LAYERS)
print(f"encoder width {D_ENC} | decoder width {D_DEC}, {N_LAYERS} layers | start {START}, end {EOS}")

## Cell 5 — Bridge, LoRA, and the prefix forward pass

Each training row is `[<bos>] + prefix + target + <end_of_turn>` as input embeddings; the loss is on the target only. Translation: Gemma's `generate()` cannot start from embeddings, so it gets `<bos>` + one placeholder token per prefix vector, and a hook on Gemma's embedding layer swaps the placeholders' embeddings for the prefix on the first forward pass (`translate` checks that the swap happened).

**Split roles.** `STAGE_GROUPS` names what each stage trains; `set_trainable(stage)` freezes everything else, and `make_optimizer` only receives the stage's groups. Gemma's LoRA starts at B = 0, so while it is not trained Gemma behaves exactly as released.

In [ ]:
# Cell 5 — Bridge (MLP into Gemma's embedding space), LoRA, the prefix forward pass, translation

import torch.nn.functional as F
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


class Bridge(nn.Module):
    """Encoder states -> vectors in Gemma's input-embedding space: a 2-layer MLP, then RMSNorm at the embeddings' scale."""

    def __init__(self, d_in, d_out):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(d_in, d_out), nn.GELU(), nn.Linear(d_out, d_out))
        self.norm = nn.RMSNorm(d_out)

    def forward(self, h):
        return self.norm(self.mlp(h))


torch.manual_seed(SEED)
bridge = Bridge(D_ENC, D_DEC).to(ENC_DEVICE)
decoder = prepare_model_for_kbit_training(decoder, use_gradient_checkpointing=True,
                                          gradient_checkpointing_kwargs={"use_reentrant": False})
PROJ = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
dec_model = get_peft_model(decoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                               target_modules=rf".*language_model.*\.({'|'.join(PROJ)})"))
encoder = get_peft_model(encoder, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["Wqkv", "Wo", "Wi"]))
EMBED = dec_model.get_input_embeddings()         # Gemma's own token embeddings (with Gemma's scaling)

with torch.no_grad():                             # start the prefix at the scale of real token embeddings
    sample = dec_tok(" ".join(r["target"] for r in TRAIN[:50]), add_special_tokens=False)["input_ids"]
    EMB_RMS = EMBED(torch.tensor(sample, device=DEC_DEVICE)).float().pow(2).mean(-1).sqrt().mean().item()
    bridge.norm.weight.fill_(EMB_RMS)

GROUPS = {"encoder LoRA": [p for n, p in encoder.named_parameters() if "lora_" in n],
          "bridge": list(bridge.parameters()),
          "decoder LoRA": [p for n, p in dec_model.named_parameters() if "lora_" in n]}
for params in GROUPS.values():
    for p in params:
        p.requires_grad_(True)
LRS = {"encoder LoRA": 1e-4, "bridge": 3e-4, "decoder LoRA": 5e-5}

S1_DECODER_LORA = True                            # R14: with False (Gemma frozen) stage 1 never clicked; reset before stage 2
STAGE_GROUPS = {1: ["encoder LoRA", "bridge"] + (["decoder LoRA"] if S1_DECODER_LORA else []),
                2: ["decoder LoRA"]}


def set_trainable(stage):
    """Only the stage's groups receive gradients; the rest are frozen."""
    for g, params in GROUPS.items():
        for p in params:
            p.requires_grad_(g in STAGE_GROUPS[stage])


def train_mode(stage):
    """Encoder side: train mode only while it is trained (dropout). Gemma always in train mode: its gradient
    checkpointing only runs in train mode, and it has no dropout of its own (an untrained LoRA, B = 0, adds nothing)."""
    encoder.train(stage == 1)
    bridge.train(stage == 1)
    dec_model.train()


def reset_decoder_lora():
    """Gemma's LoRA back to its initial state (B = 0, so Gemma as released): stage 2 learns the voice from scratch."""
    for n, p in dec_model.named_parameters():
        if "lora_B" in n:
            p.data.zero_()


def prefixes(texts):
    """Encoder input texts -> one prefix per text: bridge vectors for its real (unpadded) tokens, on GPU 1."""
    t = enc_tok(texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(ENC_DEVICE)
    h = encoder(input_ids=t["input_ids"], attention_mask=t["attention_mask"]).last_hidden_state
    p = bridge(h).to(DEC_DEVICE)
    return [p[i, :int(n)] for i, n in enumerate(t["attention_mask"].sum(1))]


def embed_ids(ids):
    return EMBED(torch.tensor(ids, device=DEC_DEVICE))


def batch_embeds(records, with_target=True):
    """Each row: [<bos>] + prefix + (target + <end_of_turn>), right-padded. Labels on the target only."""
    rows, labels = [], []
    for r, p in zip(records, prefixes([r["enc_input"] for r in records])):
        head = torch.cat([embed_ids([START]), p])
        if with_target:
            tgt = dec_tok(r["target"], add_special_tokens=False)["input_ids"][:96] + [EOS]
            rows.append(torch.cat([head, embed_ids(tgt)]))
            labels.append([-100] * len(head) + tgt)
        else:
            rows.append(head)
    T = max(len(x) for x in rows)
    emb = torch.stack([F.pad(x, (0, 0, 0, T - len(x))) for x in rows])
    mask = torch.tensor([[1] * len(x) + [0] * (T - len(x)) for x in rows], device=DEC_DEVICE)
    lab = torch.tensor([y + [-100] * (T - len(y)) for y in labels], device=DEC_DEVICE) if with_target else None
    return emb, mask, lab


def loss_on(records):
    emb, mask, lab = batch_embeds(records)
    return dec_model(inputs_embeds=emb, attention_mask=mask, labels=lab, token_type_ids=torch.zeros_like(mask)).loss


class PrefixSlot:
    """Prefix vectors waiting to be written into the next full-prompt embedding (used by translate only)."""
    vectors = None


SLOT = PrefixSlot()
PLACEHOLDER = dec_tok(" .", add_special_tokens=False)["input_ids"][-1]


def write_prefix(module, args, output):
    """Gemma's generate() cannot start from embeddings, so translate() passes <bos> + placeholder tokens and this
    hook replaces the placeholders' embeddings with the prefix, on the first (full-prompt) forward pass only."""
    if SLOT.vectors is not None and output.dim() == 3 and output.shape[1] == 1 + len(SLOT.vectors):
        output = output.clone()
        output[:, 1:] = SLOT.vectors.to(output.dtype)
        SLOT.vectors = None
    return output


EMBED.register_forward_hook(write_prefix)


@torch.no_grad()
def translate(r):
    dec_model.eval()
    encoder.eval()
    p = prefixes([r["enc_input"]])[0]
    ids = torch.tensor([[START] + [PLACEHOLDER] * len(p)], device=DEC_DEVICE)
    SLOT.vectors = p
    try:
        out = dec_model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), do_sample=False, num_beams=1,
                                 max_new_tokens=MAX_NEW_TOKENS, eos_token_id=STOPS, pad_token_id=PAD,
                                 temperature=None, top_p=None, top_k=None)
    finally:
        written = SLOT.vectors is None
        SLOT.vectors = None
    assert written, "the prefix was not written into the prompt"
    new = out[0, ids.shape[1]:].tolist()
    return {"output": dec_tok.decode(new, skip_special_tokens=True).strip(), "stopped": bool(new) and new[-1] in STOPS}


@torch.no_grad()
def mean_loss(records, batch=8):
    dec_model.eval()
    encoder.eval()
    total = 0.0
    for i in range(0, len(records), batch):
        chunk = records[i:i + batch]
        total += loss_on(chunk).item() * len(chunk)
    return round(total / len(records), 4)


def make_optimizer(steps, stage):
    opt = torch.optim.AdamW([{"params": GROUPS[g], "lr": LRS[g]} for g in STAGE_GROUPS[stage]], weight_decay=0.0)
    return opt, torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.0, 1 - s / max(1, steps)))


def snapshot():
    return [p.detach().to("cpu", copy=True) for params in GROUPS.values() for p in params]


def restore(state):
    for p, s in zip([p for params in GROUPS.values() for p in params], state):
        p.data.copy_(s.to(p.device))


def chrf(hyp, ref, n_max=6, beta=2.0):
    h, r = hyp.replace(" ", ""), ref.replace(" ", "")
    ps, rs = [], []
    for n in range(1, n_max + 1):
        hc = Counter(h[i:i + n] for i in range(len(h) - n + 1))
        rc = Counter(r[i:i + n] for i in range(len(r) - n + 1))
        if hc and rc:
            m = sum((hc & rc).values())
            ps.append(m / sum(hc.values()))
            rs.append(m / sum(rc.values()))
    if not ps:
        return 0.0
    p, q = statistics.mean(ps), statistics.mean(rs)
    return 100 * (1 + beta ** 2) * p * q / (beta ** 2 * p + q) if p + q else 0.0


def evaluate(records):
    rows = []
    for r in records:
        out = translate(r)
        rows.append({**{k: r.get(k) for k in ("id", "persona", "corpus", "source", "enc_input", "references")}, **out,
                     "chrf": max(chrf(out["output"], ref) for ref in r["references"])})
    score = {"n": len(rows), "chrf": round(statistics.mean(x["chrf"] for x in rows), 1),
             "stopped": round(sum(x["stopped"] for x in rows) / len(rows), 3),
             "distinct_outputs": len({x["output"] for x in rows})}
    return rows, score


def show(rows, n, seed=7):
    for x in random.Random(seed).sample(rows, min(n, len(rows))):
        print(f"  in : {x['enc_input']}\n  out: {x['output']}" + ("" if x["stopped"] else "  [did not stop]") +
              f"\n  ref: {x['references'][0]}\n")


print({k: f"{sum(p.numel() for p in v) / 1e6:.2f} M" for k, v in GROUPS.items()}, f"| prefix scale {EMB_RMS:.3f}")
print("before training:", translate(VAL[0])["output"][:120])

## Cell 6 — Stage 1: reconstruction through the bridge

**Trains the encoder LoRA and the bridge only; Gemma is frozen** (unless `S1_DECODER_LORA`). The gradient still flows *through* Gemma to reach the prefix, so a step costs about the same as before. One pass over `CORPUS_TOTAL` sentences (default 12,000), batches of 8, linear decay. Every 500 steps: held-out loss (400 pairs), **chrF on 40 held-out sentences with two examples** (has reconstruction clicked yet? With Gemma frozen it may take longer than R10's step 1,000), and a **checkpoint** (`/kaggle/working/stage1_ckpt.pt`: weights, optimizer, step). If the kernel dies, restart it and run all cells again: this cell resumes from the last checkpoint instead of step 0. Skipped when `CORPUS_TOTAL` is 0. After the pass: chrF, stopping and distinct outputs on 120 held-out pairs (60 from each corpus), with examples. The stage-1 weights are saved to `/kaggle/working/stage1.pt`.

In [ ]:
# Cell 6 — Stage 1

S1_BATCH, S1_EVAL_EVERY = 8, 500
S1_CKPT = f"{WORK}/stage1_two_paths_ckpt.pt"    # written every S1_EVAL_EVERY steps; a restarted kernel resumes from it
s1_steps = -(-len(CORPUS_TRAIN) // S1_BATCH)
set_trainable(1)
opt, sched = make_optimizer(s1_steps, 1)
S1_QUICK = [r for r in CORPUS_HELD if r["corpus"] == "top"][:20] + [r for r in CORPUS_HELD if r["corpus"] == "phinc"][:20]
start = 0
ck = torch.load(S1_CKPT, weights_only=False) if os.path.exists(S1_CKPT) else None
if ck is not None and (ck["n_pairs"], ck["groups"]) != (len(CORPUS_TRAIN), STAGE_GROUPS[1]):
    print("ignoring a checkpoint from a different stage 1 (corpus size or trained parts differ); starting at step 0")
    ck = None
if ck is not None:
    restore(ck["params"])
    opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"])
    S1_LOG, start = ck["log"], ck["step"]
    del ck
    gc.collect()
    print(f"resuming stage 1 from the checkpoint at step {start}")
else:
    S1_LOG = [{"step": 0, "held_out_loss": mean_loss(CORPUS_HELD)}]
print(S1_LOG[-1])
torch.cuda.reset_peak_memory_stats(DEC_DEVICE)
t0, run = time.perf_counter(), []
for step in range(start + 1, s1_steps + 1):
    train_mode(1)
    loss = loss_on(CORPUS_TRAIN[(step - 1) * S1_BATCH: step * S1_BATCH])
    loss.backward()
    opt.step()
    sched.step()
    opt.zero_grad()
    run.append(loss.item())
    if step % S1_EVAL_EVERY == 0 or step == s1_steps:
        quick_rows, quick = evaluate(S1_QUICK)
        S1_LOG.append({"step": step, "train_loss": round(sum(run) / len(run), 4),
                       "held_out_loss": mean_loss(CORPUS_HELD), "chrf_40": quick["chrf"]})
        run = []
        torch.save({"step": step, "n_pairs": len(CORPUS_TRAIN), "groups": STAGE_GROUPS[1], "params": snapshot(),
                    "opt": opt.state_dict(),
                    "sched": sched.state_dict(), "log": S1_LOG}, S1_CKPT)
        gc.collect()
        torch.cuda.empty_cache()
        print(S1_LOG[-1], f"| {time.perf_counter() - t0:.0f}s | peak GPU 1 "
              f"{torch.cuda.max_memory_allocated(DEC_DEVICE) / 2**30:.1f} GB | checkpoint saved")
        for x in quick_rows[:1] + quick_rows[20:21]:
            print(f"    {x['source'][:60]:60s} -> {x['output'][:60]}")

torch.save(snapshot(), f"{WORK}/stage1.pt")
S1_PROBE = [r for r in CORPUS_HELD if r["corpus"] == "top"][:60] + [r for r in CORPUS_HELD if r["corpus"] == "phinc"][:60]
S1_ROWS, S1_SCORE = evaluate(S1_PROBE) if CORPUS_TRAIN else ([], None)
print("\nstage 1, held-out corpus pairs:", S1_SCORE)
if S1_ROWS:
    show(S1_ROWS, 8)

## Cell 7 — Stage 2, one path at a time

`run_path(path)` starts from the stage-1 weights (`STAGE1_STATE`), resets Gemma's LoRA for `reset`, trains Gemma's LoRA alone (micro-batches of 4, 2-step accumulation, up to 6 epochs, early stopping after 2 epochs without a lower `validation_daily` loss, best epoch kept), then scores every held-out split, translates the chosen sentences, samples eight variants of one sentence, saves the path's Gemma LoRA (`/kaggle/working/decoder-lora-<path>/`) and its results (`/kaggle/working/path_<path>.json`). If that results file exists, the path is skipped and its results are reloaded.

In [ ]:
# Cell 7 — Stage 2 for one path; evaluation, chosen sentences, samples

MAX_EPOCHS, PATIENCE, MICRO, ACCUM = 6, 2, 4, 2
SAMPLE_SENTENCE, N_SAMPLES, SAMPLE_TEMPERATURE = "I'll be late to the office because of traffic.", 8, 0.9
STAGE1_STATE = snapshot()                   # both paths start from exactly these weights
norm = lambda t: re.sub(r"\W+", " ", t.lower()).strip()
train_targets = {r["target"].strip() for r in TRAIN}


@torch.no_grad()
def translate_sampled(sentence, n, temperature, top_p=0.95):
    dec_model.eval()
    encoder.eval()
    p = prefixes([f"{NAME['thanos']}: {sentence}"])[0]
    ids = torch.tensor([[START] + [PLACEHOLDER] * len(p)], device=DEC_DEVICE)
    outs = []
    for _ in range(n):
        SLOT.vectors = p
        out = dec_model.generate(input_ids=ids, attention_mask=torch.ones_like(ids), do_sample=True,
                                 temperature=temperature, top_p=top_p, max_new_tokens=MAX_NEW_TOKENS,
                                 eos_token_id=STOPS, pad_token_id=PAD)
        assert SLOT.vectors is None, "the prefix was not written into the prompt"
        outs.append(dec_tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip())
    return outs


def run_path(path):
    out_file = f"{WORK}/path_{path}.json"
    if os.path.exists(out_file):
        print(f"[{path}] already done; results reloaded from {out_file}")
        return json.load(open(out_file))
    restore(STAGE1_STATE)
    if path == "reset":
        reset_decoder_lora()
    set_trainable(2)
    opt, sched = make_optimizer(MAX_EPOCHS * -(-len(TRAIN) // (MICRO * ACCUM)), 2)
    samples = [VAL[0], VAL[4], VAL[8]]
    log = [{"epoch": 0, "validation_loss": mean_loss(VAL)}]
    print(f"[{path}]", log[-1])
    for r in samples:
        print(f"    {r['enc_input'][:70]:70s} -> {translate(r)['output']}")
    best = (log[0]["validation_loss"], 0, snapshot())
    since_best, rng, t0 = 0, random.Random(SEED), time.perf_counter()
    for epoch in range(1, MAX_EPOCHS + 1):
        train_mode(2)
        order = rng.sample(TRAIN, len(TRAIN))
        batches = [order[i:i + MICRO] for i in range(0, len(order), MICRO)]
        run = []
        for n, b in enumerate(batches, 1):
            loss = loss_on(b)
            (loss / ACCUM).backward()
            run.append(loss.item())
            if n % ACCUM == 0 or n == len(batches):
                opt.step()
                sched.step()
                opt.zero_grad()
        log.append({"epoch": epoch, "train_loss": round(sum(run) / len(run), 4), "validation_loss": mean_loss(VAL)})
        print(f"[{path}]", log[-1], f"| {time.perf_counter() - t0:.0f}s")
        for r in samples:
            print(f"    {r['enc_input'][:70]:70s} -> {translate(r)['output']}")
        if log[-1]["validation_loss"] < best[0]:
            best, since_best = (log[-1]["validation_loss"], epoch, snapshot()), 0
        else:
            since_best += 1
        if since_best >= PATIENCE:
            print(f"[{path}] early stop: no lower validation loss for {PATIENCE} epochs")
            break
    restore(best[2])
    print(f"[{path}] kept epoch {best[1]} (validation loss {best[0]})")
    del best, opt, sched
    gc.collect()
    torch.cuda.empty_cache()

    results, scores = {}, {}
    for split in EVAL_SPLITS:
        results[split], scores[split] = evaluate(SPLITS[split])
        scores[split]["copies_of_training_targets"] = sum(x["output"] in train_targets for x in results[split])
        scores[split]["copies_of_source"] = sum(norm(x["output"]) == norm(x["source"]) for x in results[split])
        print(f"[{path}]", split, scores[split])
    demo = [{"input": s, **translate({"enc_input": f"{NAME['thanos']}: {s}"})} for s in DEMO_INPUTS]
    sampled = translate_sampled(SAMPLE_SENTENCE, N_SAMPLES, SAMPLE_TEMPERATURE)
    dec_model.save_pretrained(f"{WORK}/decoder-lora-{path}")
    res = {"path": path, "stage2_log": log, "kept_epoch": [e["epoch"] for e in log if e["validation_loss"] == min(x["validation_loss"] for x in log)][0],
           "scores": scores, "results": results, "demo": demo, "sampled": sampled}
    json.dump(res, open(out_file, "w"), ensure_ascii=False)
    print(f"\n[{path}] examples:")
    show([x for s in EVAL_SPLITS for x in results[s]], 8)
    print(f"[{path}] chosen sentences:")
    for k, d in enumerate(demo, 1):
        print(f"{k:2d}. in : {d['input']}\n    out: {d['output']}" + ("" if d["stopped"] else "   [did not stop]"))
    print(f"\n[{path}] {N_SAMPLES} samples at temperature {SAMPLE_TEMPERATURE}: {SAMPLE_SENTENCE}")
    for o in sampled:
        print("   -", o)
    return res


PATHS = {}

## Cell 8 — Path A: carry (stage 2 continues from the stage-1 Gemma LoRA)

In [ ]:
# Cell 8 — Path A: carry

PATHS["carry"] = run_path("carry")

## Cell 9 — Path B: reset (stage 2 from a zeroed Gemma LoRA)

In [ ]:
# Cell 9 — Path B: reset

PATHS["reset"] = run_path("reset")

## Cell 10 — Side by side

Scores per split and path, the kept epoch, then each chosen sentence with both paths' outputs, then both paths' sampled variants.

In [ ]:
# Cell 10 — Carry vs reset

cols = ["chrf", "stopped", "distinct_outputs", "copies_of_source"]
print(f"{'split':18s} {'path':6s} " + " ".join(f"{c:>17s}" for c in cols))
for split in EVAL_SPLITS:
    for path in ("carry", "reset"):
        s = PATHS[path]["scores"][split]
        print(f"{split:18s} {path:6s} " + " ".join(f"{s[c]:>17}" for c in cols))
print("\nkept epoch:", {p: PATHS[p]["kept_epoch"] for p in PATHS},
      "| best validation_daily loss:", {p: min(e["validation_loss"] for e in PATHS[p]["stage2_log"]) for p in PATHS})
print("\nchosen sentences:")
for a, b in zip(PATHS["carry"]["demo"], PATHS["reset"]["demo"]):
    print(f"in   : {a['input']}\ncarry: {a['output']}\nreset: {b['output']}\n")
for p in ("carry", "reset"):
    print(f"{p} samples ({SAMPLE_SENTENCE}):")
    for o in PATHS[p]["sampled"]:
        print("   -", o)

## Cell 11 — Save, settings, records

The encoder LoRA and the bridge are shared by both paths (frozen in stage 2); each path's Gemma LoRA is already saved (`decoder-lora-carry/`, `decoder-lora-reset/`). Download them from `/kaggle/working/` if the result is worth keeping.

In [ ]:
# Cell 11 — Save the shared parts; settings and records

encoder.save_pretrained(f"{WORK}/encoder-lora")
torch.save({"bridge": bridge.state_dict(), "prefix_scale": EMB_RMS}, f"{WORK}/bridge.pt")
print(json.dumps({"revisions": REV, "corpus": {"train": dict(Counter(r["corpus"] for r in CORPUS_TRAIN)),
                                               "held_out": len(CORPUS_HELD), "dropped": dict(dropped)},
                  "lrs": LRS, "stage_groups": STAGE_GROUPS,
                  "stage1": {"batch": S1_BATCH, "log": S1_LOG, "score": S1_SCORE},
                  "paths": {p: {"stage2_log": PATHS[p]["stage2_log"], "kept_epoch": PATHS[p]["kept_epoch"],
                                "scores": PATHS[p]["scores"]} for p in PATHS}}, indent=1))
print("\n----- RECORDS (JSON lines) -----")
for x in S1_ROWS:
    print(json.dumps({"split": "corpus_held_out", **x}, ensure_ascii=False))
for p in PATHS:
    for split, rows in PATHS[p]["results"].items():
        for x in rows:
            print(json.dumps({"path": p, "split": split, **x}, ensure_ascii=False))
print("----- END RECORDS -----")